# ASR Pipeline — interactive exploration

Run each stage of the pipeline on its own. Tweak knobs, re-run a stage, accept its
output, move on to the next.

```
Diarization → Routing → SE (solo) → SS (overlap) → Assembly → Transcription
```

**Workflow:**

1. Cell 1: imports + default `INPUT_PATH`.
2. Cell 2 (optional): drag an audio file onto the widget. Updates `INPUT_PATH`.
3. Cell 3: build `cfg`, construct `pipeline`, load audio into `ctx`.
4. Cells 4 onwards — one per stage. Each cell has a knob block at the top, a
   `pipeline.run_stage(...)` call, and an inspection block. Re-run the cell as
   many times as you like; the stage's model stays loaded between runs.
5. When you switch to a different stage, the previous model is freed and the
   new one loads (one-model-on-GPU invariant — same as the POC).
6. Final cell: `pipeline.unload()` releases the last loaded model.

**Prerequisites:** `pyannote.audio`, `whisper`, `speechbrain`, MP-SENet checkpoint
+ `config.json`, the SepFormer checkpoint pointed to by `default.yaml`.

In [ ]:
# --- Imports + path bootstrapping ---
import os, sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "asr":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)   # so relative checkpoint paths resolve

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from IPython.display import Audio, display

# Edit for a different input recording (overridden by the drag-and-drop in cell 2).
INPUT_PATH = "/tmp/clarin_60s.wav"

print("Project root:", PROJECT_ROOT)
print("Input       :", INPUT_PATH, "(exists:", Path(INPUT_PATH).exists(), ")")

In [ ]:
# --- Drag-and-drop audio loader (optional) -------------------------------
# Drop an audio file onto the widget (or click to pick one). On upload,
# the file is saved to /tmp and `INPUT_PATH` is updated. If you don't use
# the widget, the path set in cell 1 stays in effect.
import tempfile
import ipywidgets as widgets

uploader = widgets.FileUpload(
    accept=".wav,.mp3,.flac,.ogg,.m4a",
    multiple=False,
    description="Drop audio here",
    layout=widgets.Layout(width="320px"),
)
status = widgets.HTML(value=f"<i>Using path from cell 1: <code>{INPUT_PATH}</code></i>")


def _on_upload(change):
    global INPUT_PATH
    val = uploader.value
    if not val:
        return
    if isinstance(val, dict):
        name = next(iter(val))
        content = val[name]["content"]
    else:
        item = val[0]
        name = item["name"]
        content = item["content"]
    out = Path(tempfile.gettempdir()) / f"upload_{name}"
    out.write_bytes(bytes(content))
    INPUT_PATH = str(out)
    status.value = (
        f"<b>Uploaded:</b> <code>{name}</code> &rarr; "
        f"<code>{INPUT_PATH}</code> &nbsp;({len(content)/1024:.1f} KiB)"
    )


uploader.observe(_on_upload, names="value")
display(widgets.VBox([uploader, status]))

In [ ]:
# --- Build config, construct pipeline, load audio ---
from asr_pipeline import Pipeline
from asr_pipeline.config import load_pipeline_config_from_yaml

cfg = load_pipeline_config_from_yaml("asr_pipeline/configs/default.yaml")

# Optional: persist intermediates to disk in addition to the in-memory `ctx`.
# cfg.artifact_dir = "runs/notebook"
# cfg.spill_intermediate = True
cfg.__post_init__()

pipeline = Pipeline(cfg)
ctx = pipeline.load_audio(INPUT_PATH)
print(pipeline)
print(f"Audio loaded: {ctx.audio.shape[0]/ctx.sample_rate:.2f}s @ {ctx.sample_rate}Hz")

## Stage 1 — Diarization

Per-speaker bars + red highlights for pyannote-detected overlaps.

Re-run this cell after editing the knobs to redo diarization without touching
other stages. The pyannote model stays loaded between runs.

In [ ]:
# --- Knobs ---
# cfg.diarization.num_speakers = 2

pipeline.run_stage("diarization", ctx)


# --- Inspect ---
def plot_diarization(seg_df, ovl_df, total_dur, title="Diarization"):
    if len(seg_df) == 0:
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.text(0.5, 0.5, "No speakers detected", ha="center", va="center")
        return fig
    speakers = sorted(seg_df["speaker"].unique())
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}
    fig, ax = plt.subplots(figsize=(16, max(2.5, len(speakers) + 1)))
    for _, row in seg_df.iterrows():
        ax.barh(spk_to_y[row["speaker"]], row["duration"], left=row["start"],
                height=0.6, color=spk_color[row["speaker"]], alpha=0.85)
    for _, row in ovl_df.iterrows():
        ax.axvspan(row["start"], row["end"], color="#e74c3c", alpha=0.25)
    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(speakers, fontweight="bold")
    ax.set_xlim(0, total_dur); ax.set_xlabel("Time (s)"); ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3, linestyle="--")
    handles = [mpatches.Patch(color=spk_color[s], label=s) for s in speakers]
    handles.append(mpatches.Patch(color="#e74c3c", alpha=0.4, label="overlap"))
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title, fontweight="bold"); plt.tight_layout()
    return fig

plot_diarization(
    ctx.diarization.segments_df,
    ctx.diarization.overlaps_df,
    ctx.diarization.total_duration_s,
)
plt.show()
print(f"{len(ctx.diarization.segments_df)} segments, "
      f"{len(ctx.diarization.overlaps_df)} overlaps")
display(ctx.diarization.segments_df.head(10))

## Stage 2 — Routing (overlap selection)

Decide which intervals get sent to SepFormer: filter pyannote's overlap
timeline by `min_overlap_dur`, then merge regions closer than `merge_gap`
so SepFormer sees one contiguous call per cluster.

Per-speaker solos are **not** produced here; the assembler in Stage 4
derives them on the fly by subtracting these overlap regions from
pyannote's per-speaker segments.

In [ ]:
# --- Knobs ---
cfg.routing.min_overlap_dur = 0.20
cfg.routing.merge_gap       = 0.20

pipeline.run_stage("routing", ctx)


# --- Inspect: overlay diarization (background) + routing overlap regions (red) ---
def plot_routing(seg_df, overlap_regions, total_dur, title="Routing — overlap selection"):
    """Per-speaker pyannote segments + red strips for ctx.overlap_regions.

    No solo bars — the assembler derives per-speaker solos on the fly by
    subtracting overlap_regions from pyannote segments. The visualisation
    shows: (a) what pyannote saw per speaker, (b) which slices will be
    sent to SepFormer.
    """
    if len(seg_df) == 0:
        fig, ax = plt.subplots(figsize=(12, 2))
        ax.text(0.5, 0.5, "No speakers detected", ha="center", va="center")
        return fig
    speakers = sorted(seg_df["speaker"].unique())
    spk_to_y = {s: i for i, s in enumerate(speakers)}
    spk_to_label = {s: chr(ord("A") + i) for i, s in enumerate(speakers)}
    cmap = plt.cm.get_cmap("Set2", max(len(speakers), 3))
    spk_color = {s: cmap(i) for i, s in enumerate(speakers)}

    fig, ax = plt.subplots(figsize=(16, max(2.5, len(speakers) + 1)))

    # Foreground: pyannote per-speaker segments (unpadded).
    for _, row in seg_df.iterrows():
        ax.barh(
            spk_to_y[row["speaker"]],
            row["duration"],
            left=row["start"],
            height=0.6,
            color=spk_color[row["speaker"]],
            alpha=0.85,
        )

    # Background: overlap regions sent to SepFormer (post-merge).
    for s, e in overlap_regions:
        ax.axvspan(s, e, color="#e74c3c", alpha=0.25)

    ax.set_yticks(range(len(speakers)))
    ax.set_yticklabels(
        [f"{spk_to_label[s]}: {s}" for s in speakers], fontweight="bold"
    )
    ax.set_xlim(0, total_dur)
    ax.set_xlabel("Time (s)")
    ax.invert_yaxis()
    ax.grid(axis="x", alpha=0.3, linestyle="--")

    handles = [
        mpatches.Patch(color=spk_color[s], label=spk_to_label[s])
        for s in speakers
    ]
    handles.append(mpatches.Patch(color="#e74c3c", alpha=0.4, label="→ SepFormer"))
    ax.legend(handles=handles, loc="upper right")
    ax.set_title(title, fontweight="bold")
    plt.tight_layout()
    return fig


plot_routing(
    ctx.diarization.segments_df,
    ctx.overlap_regions,
    ctx.diarization.total_duration_s,
)
plt.show()

total_overlap_s = sum(e - s for s, e in ctx.overlap_regions)
print(f"speakers          : {ctx.speakers}")
print(f"overlap regions   : {len(ctx.overlap_regions)} "
      f"({total_overlap_s:.2f}s total → SepFormer)")
for i, (s, e) in enumerate(ctx.overlap_regions):
    print(f"  #{i:2d}  {s:6.2f} - {e:6.2f}  ({e - s:.2f}s)")


## Stage 3a — Full-recording enhancement (MP-SENet)

One MP-SENet pass over the whole input. Recordings longer than
`max_segment_length_s` (default 8 s) are processed via Hann overlap-add.
The result lives in `ctx.enhanced_full` and is sliced per speaker at
assembly time.

Note: SepFormer in Stage 3b runs on the *original* `ctx.audio`, not on
this enhanced version — MP-SENet is a denoiser, not a separator, and
tends to suppress the quieter speaker in overlapping speech.

The MP-SENet model loads on the first run of this cell; subsequent runs
reuse the same loaded model.

In [ ]:
# --- Knobs ---
# cfg.enhancement.max_segment_length_s = 8.0   # solos longer than this are chunked

pipeline.run_stage("enhancement", ctx)


# --- Inspect: full recording before vs after enhancement ---
sr = ctx.sample_rate
raw = ctx.audio
enhanced = ctx.enhanced_full
t = np.arange(len(raw)) / sr

fig, axes = plt.subplots(2, 1, figsize=(16, 4), sharex=True, sharey=True)
axes[0].plot(t, raw, linewidth=0.4, color="C0")
axes[0].set_ylabel("raw")
axes[0].grid(alpha=0.3)
axes[1].plot(t, enhanced, linewidth=0.4, color="C1")
axes[1].set_ylabel("enhanced")
axes[1].set_xlabel("Time (s)")
axes[1].grid(alpha=0.3)
axes[0].set_title("Full-recording enhancement", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"duration : {len(raw)/sr:.2f}s @ {sr}Hz")
print(f"chunking : > {cfg.enhancement.max_segment_length_s:.1f}s → "
      f"Hann overlap-add ({'yes' if len(raw)/sr > cfg.enhancement.max_segment_length_s else 'no — one shot'})")

print("\nraw:")
display(Audio(raw, rate=sr))
print("enhanced:")
display(Audio(enhanced, rate=sr))


## Stage 3b — Overlap separation (SepFormer + VAD)

For each overlap region:

- top: mixture (16 kHz)
- middle two: separator outputs s1 / s2 — raw (translucent) vs VAD-gated
- 4th panel: per-stream binary VAD **mask** (after soft threshold + dilation)
- 5th panel: raw silero **VAD probabilities** per 32 ms frame, with
  horizontal lines at `vad_threshold` (upper, "definitely speech") and
  `vad_soft_threshold` (Schmitt-trigger lower bound) — read off which
  frames cross which cutoff to see *why* the mask came out the way it did
- green tint: the **emit region** that the assembler will splice into the
  per-speaker stream (controlled by `cfg.separation.seam_mode`)

Try editing one of the knobs (e.g. switch `context_window_mode` to `fixed_pad`)
and re-running this cell to see how the pad/emit regions move.

In [ ]:
# --- Knobs ---
cfg.separation.context_window_mode       = "snap_to_vad"        # snap_to_vad | fixed_pad | none
cfg.separation.context_pad_seconds       = 1.0
cfg.separation.min_fragment_length_s     = 4.0                  # bump short windows up to this
cfg.separation.seam_mode                 = "snap_to_silence"    # zero_crossing | overlap_boundary | snap_to_silence
cfg.separation.seam_search_radius_s      = 0.05                 # zero-crossing nudge radius
cfg.separation.snap_silence_max_extend_s = 0.30                 # snap_to_silence: max outward extension (s)
cfg.separation.volume_normalization      = "sum_equals_mix"     # sum_equals_mix | none
cfg.separation.vad_threshold             = 0.25                 # upper ("definitely speech")
cfg.separation.vad_soft_threshold        = 0.10                 # Schmitt-trigger lower (0 = disable)
cfg.separation.vad_attack_frames         = 1                    # 32 ms each
cfg.separation.vad_release_frames        = 1

pipeline.run_stage("separation", ctx)


# --- Inspect ---
sr = ctx.sample_rate
if not ctx.overlap_separated:
    print("No overlap regions in this recording.")
else:
    for ovl in ctx.overlap_separated:
        print(f"\nOverlap #{ovl['idx']} — "
              f"orig=[{ovl['start']:.2f}, {ovl['end']:.2f}]  "
              f"pad=[{ovl['pad_start']:.2f}, {ovl['pad_end']:.2f}]  "
              f"emit=[{ovl['emit_start']:.2f}, {ovl['emit_end']:.2f}]  "
              f"chunked={ovl['chunked']}  vol_scale={ovl['volume_scale']:.3f}")

        t = np.arange(len(ovl["mix"])) / sr
        # Frame times for the raw VAD probabilities (silero window = 512 samples).
        FRAME_N = 512
        frame_t1 = np.arange(len(ovl["probs1"])) * FRAME_N / sr
        frame_t2 = np.arange(len(ovl["probs2"])) * FRAME_N / sr

        fig, axes = plt.subplots(5, 1, figsize=(14, 8), sharex=True)
        axes[0].plot(t, ovl["mix"], lw=0.5, color="C0");  axes[0].set_ylabel("mix")
        axes[1].plot(t, ovl["s1_raw"], lw=0.5, color="C1", alpha=0.4, label="raw")
        axes[1].plot(t, ovl["s1_gated"], lw=0.5, color="C1", label="gated")
        axes[1].set_ylabel("s1"); axes[1].legend(fontsize=8, loc="upper right")
        axes[2].plot(t, ovl["s2_raw"], lw=0.5, color="C2", alpha=0.4, label="raw")
        axes[2].plot(t, ovl["s2_gated"], lw=0.5, color="C2", label="gated")
        axes[2].set_ylabel("s2"); axes[2].legend(fontsize=8, loc="upper right")
        axes[3].plot(t, ovl["mask1"], color="C1", label="mask 1")
        axes[3].plot(t, ovl["mask2"], color="C2", label="mask 2", alpha=0.7)
        axes[3].set_ylabel("mask"); axes[3].set_ylim(-0.1, 1.1)
        axes[3].legend(fontsize=8, loc="upper right")
        # Raw silero probabilities (step plot makes the 32 ms frame discretization
        # visible). Thresholds drawn as horizontal lines so you can read off
        # which frames are above/below the upper/soft-threshold cutoff.
        axes[4].step(frame_t1, ovl["probs1"], where="post", color="C1", label="probs 1")
        axes[4].step(frame_t2, ovl["probs2"], where="post", color="C2", label="probs 2", alpha=0.7)
        axes[4].axhline(cfg.separation.vad_threshold, color="k", ls="--", lw=0.7,
                        alpha=0.6, label=f"upper={cfg.separation.vad_threshold}")
        if 0 < cfg.separation.vad_soft_threshold < cfg.separation.vad_threshold:
            axes[4].axhline(cfg.separation.vad_soft_threshold, color="k", ls=":",
                            lw=0.7, alpha=0.6,
                            label=f"soft={cfg.separation.vad_soft_threshold}")
        axes[4].set_ylabel("VAD probs"); axes[4].set_ylim(-0.05, 1.05)
        axes[4].legend(fontsize=8, loc="upper right")
        axes[4].set_xlabel("s in padded window")
        emit_lo = ovl["emit_start"] - ovl["pad_start"]
        emit_hi = ovl["emit_end"]   - ovl["pad_start"]
        # Dotted lines show pyannote's original overlap boundary — when
        # snap_to_silence is in use, the green emit region should sit
        # outside them.
        orig_lo = ovl["start"] - ovl["pad_start"]
        orig_hi = ovl["end"]   - ovl["pad_start"]
        for ax in axes:
            ax.axvspan(emit_lo, emit_hi, color="green", alpha=0.08)
            ax.axvline(orig_lo, color="black", ls=":", lw=0.7, alpha=0.5)
            ax.axvline(orig_hi, color="black", ls=":", lw=0.7, alpha=0.5)
            ax.grid(alpha=0.3)
        plt.tight_layout(); plt.show()

        print("mix:");      display(Audio(ovl["mix"], rate=sr))
        print("s1 gated:"); display(Audio(ovl["s1_gated"], rate=sr))
        print("s2 gated:"); display(Audio(ovl["s2_gated"], rate=sr))


## Stage 4 — Per-speaker assembly

Stream view in original recording time. Blue = solo (enhanced), red = overlap
(separated). Audio players play each assembled per-speaker stream.

In [ ]:
# --- Knobs ---
cfg.assembly.output_mode              = "full_length"   # shortened | full_length
cfg.assembly.silence_separator_s      = 0.3
cfg.assembly.crossfade_ms             = 5.0             # internal piece-to-piece seam fade
cfg.assembly.edge_fade_ms             = 2.0             # stream's outermost edge fade
cfg.assembly.overlap_rms_match_solo   = True            # scale overlap pieces to per-speaker solo RMS
# cfg.assembly.per_piece_rms_norm     = True            # aggressive: normalise ALL pieces (overrides above)
# cfg.assembly.target_rms             = 0.05

pipeline.run_stage("assembly", ctx)


# --- Inspect ---
sr = ctx.sample_rate
spk_to_label = ctx.spk_to_label

fig, axes = plt.subplots(
    len(ctx.speakers), 1,
    figsize=(16, 1.2 * len(ctx.speakers) + 1),
    sharex=True, squeeze=False,
)
axes = axes[:, 0]
for ax, spk in zip(axes, ctx.speakers):
    for entry in ctx.timestamp_map.per_speaker[spk]:
        color = "#1f77b4" if entry.kind == "solo" else "#e74c3c"
        ax.barh(0, entry.orig_end - entry.orig_start, left=entry.orig_start,
                height=0.6, color=color, alpha=0.85)
    ax.set_yticks([0])
    ax.set_yticklabels([f"{spk_to_label[spk]}: {spk}"])
    ax.set_ylim(-0.5, 0.5)
    ax.grid(axis="x", alpha=0.3)
axes[-1].set_xlabel("orig time (s)")
axes[-1].set_xlim(0, ctx.diarization.total_duration_s)
axes[0].set_title("Assembled streams in original time  (blue = solo / SE, red = overlap / SS)")
plt.tight_layout(); plt.show()

print(f"output_mode  : {cfg.assembly.output_mode}")
print(f"weak_anchor  : {ctx.weak_anchor}")
for spk, audio in ctx.assembled.items():
    print(f"\nSpeaker {spk_to_label[spk]} ({spk}) — {len(audio)/sr:.2f}s")
    display(Audio(audio, rate=sr))


## Stage 5 — Transcripts

One Whisper call per assembled per-speaker stream. Word timestamps are
available in `ctx.transcripts[spk]['segments'][i]['words']`.

In [ ]:
# --- Knobs ---
# cfg.transcription.model_name      = "large-v3"
# cfg.transcription.language        = "pl"
# cfg.transcription.initial_prompt  = "Rozmowa po polsku."
# cfg.transcription.word_timestamps = True

pipeline.run_stage("transcription", ctx)


# --- Inspect ---
spk_to_label = ctx.spk_to_label
for spk in ctx.speakers:
    label = spk_to_label[spk]
    print(f"\n=== Speaker {label} ({spk}) ===")
    res = ctx.transcripts.get(spk, {})
    segments = res.get("segments", [])
    if not segments:
        text = res.get("text", "")
        print(text if text else "(empty)")
        continue
    for seg in segments:
        print(f"[{seg['start']:6.2f} → {seg['end']:6.2f}]  {seg['text'].strip()}")

## Iteration tips

- **Re-run one stage.** Edit the knob lines at the top of the stage's cell,
  then re-run that cell. Re-running the same stage twice is fast — the
  stage's model stays loaded.
- **Switch stages.** Just run a different stage's cell. The previous stage's
  model is freed, the new one loads. Only one model on GPU at any time.
- **Go back.** If you want to re-run an upstream stage after changing its
  knobs, just rerun its cell. Then re-run any downstream stages whose inputs
  changed.
- **Free GPU memory.** Run the final cell (`pipeline.unload()`) to free
  whatever's currently loaded. Useful before swapping checkpoints.
- **Reload after editing checkpoint paths.** If you change
  `cfg.<stage>.checkpoint_path`, call `pipeline.unload()` first — the loaded
  model is the old one. After unload, the next `run_stage` call will reload.
- **Persist artefacts.** Set `cfg.artifact_dir = "runs/notebook"` and
  `cfg.spill_intermediate = True` in cell 3 (then re-create the pipeline /
  re-run the stage) to also write each stage's outputs to disk.

In [ ]:
# Free whatever's currently loaded on the GPU.
pipeline.unload()
print(pipeline)